In [30]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import config as cfg
import os

import torch
from torch import nn
from torch.utils.data import DataLoader

from Data.data_loading import load_and_preprocess_data, create_tensor_from_dataframe, create_sequences, create_dataloaders 
from Training.train_matt import Trainer
from Training.basicEval import plotLoss, plotAccuracy, reportFinalMetrics, reportMultiFinalMetrics, plotMultiAccuracy, plotMultiLoss
from NextNet.model_split import FrameTransformer, print_model_info

from Training.customLoss import ADELoss, FDELoss, RMSELoss

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
root_dir = os.getcwd()  # Use current working directory as root
data_dir = os.path.join(root_dir, 'Data')
csv_dir = os.path.join(data_dir, 'one_csv')
csv_file = os.path.join(csv_dir, 'merge.csv')
model_dir = os.path.join(root_dir, 'Model', 'Saved_Model')
model_path = os.path.join(model_dir, 'ade_model_1s.pth')

print("Data directory: ", data_dir)
print("CSV directory: ", csv_dir)
print("CSV file: ", csv_file)


model_dir = os.path.join(root_dir, 'Model')
save_model_dir = os.path.join(model_dir, 'Saved_Model')
print("Model directory: ", model_dir)
print("Saved model directory: ", save_model_dir)


Data directory:  /home/jkabir/Deep-Learning-Project/Data
CSV directory:  /home/jkabir/Deep-Learning-Project/Data/one_csv
CSV file:  /home/jkabir/Deep-Learning-Project/Data/one_csv/merge.csv
Model directory:  /home/jkabir/Deep-Learning-Project/Model
Saved model directory:  /home/jkabir/Deep-Learning-Project/Model/Saved_Model


In [32]:

print(csv_dir)
# feauture_scaler takes X,Y,Height,Width
df, transformer_max_ids_per_frame, frame_scaler, xy_scaler = load_and_preprocess_data(csv_folder=csv_dir)

# 2. Create tensor from dataframe
all_data_tensor = create_tensor_from_dataframe(df, transformer_max_ids_per_frame)
print(f"Data tensor shape: {all_data_tensor.shape}")
# 3. Create input-output sequences
X, Y = create_sequences(all_data_tensor)

# 4. Create dataloaders
train_loader, test_loader, train_fetcher, test_fetcher = create_dataloaders(X, Y)

# 4. Load Model
model = torch.load(model_path, weights_only=False).to('cuda')

print(f"Initial Data X shape: {X.shape}, Y shape: {Y.shape}")
current_num_ids_from_data = X.size(2)
print(f"Number of IDs in current data: {current_num_ids_from_data}")

# Try to get model's expected num_ids and hidden_size
model_expected_num_ids = None
model_hidden_size = None

if hasattr(model, 'num_ids'):
    model_expected_num_ids = model.num_ids
    print(f"Loaded model.num_ids: {model_expected_num_ids}")
else:
    print("Loaded model does not have 'num_ids' attribute.")

if hasattr(model, 'hidden_size'):
    model_hidden_size = model.hidden_size
    print(f"Loaded model.hidden_size: {model_hidden_size}")
else:
    print("Loaded model does not have 'hidden_size' attribute.")

# Determine target_num_ids for padding
target_num_ids = None
assumed_hidden_size_for_inference = 64 # Default based on comments and common practice
if model_hidden_size is not None and model_hidden_size > 0:
    assumed_hidden_size_for_inference = model_hidden_size

if model_expected_num_ids is not None:
    target_num_ids = model_expected_num_ids
else:
    # Infer from common error pattern: embed_dim 1280 / hidden_size 64 = 20
    # The error message implies embed_dim for frame_attention is 1280
    frame_attention_embed_dim_from_error = 1280 
    target_num_ids = frame_attention_embed_dim_from_error // assumed_hidden_size_for_inference
    print(f"Using inferred target_num_ids: {target_num_ids} (based on error message embed_dim={frame_attention_embed_dim_from_error} and hidden_size={assumed_hidden_size_for_inference})")

if target_num_ids is not None and current_num_ids_from_data < target_num_ids:
    pad_len = target_num_ids - current_num_ids_from_data
    print(f"Padding data from {current_num_ids_from_data} to {target_num_ids} IDs (pad_len: {pad_len}).")
    
    # Pad X
    X_pad_shape = (X.size(0), X.size(1), pad_len, X.size(3))
    X_pad_tensor = torch.full(X_pad_shape, -1.0, dtype=X.dtype, device=X.device)
    X = torch.cat((X, X_pad_tensor), dim=2)
    
    # Pad Y
    Y_pad_shape = (Y.size(0), Y.size(1), pad_len, Y.size(3))
    Y_pad_tensor = torch.full(Y_pad_shape, -1.0, dtype=Y.dtype, device=Y.device)
    Y = torch.cat((Y, Y_pad_tensor), dim=2)
    
    print(f"Padded X shape: {X.shape}, Padded Y shape: {Y.shape}")
elif target_num_ids is not None and current_num_ids_from_data > target_num_ids:
    print(f"Warning: Data has {current_num_ids_from_data} IDs, but model expects {target_num_ids}. Truncation might be needed or model might misbehave.")
elif target_num_ids is not None and current_num_ids_from_data == target_num_ids:
    print(f"Data num_ids ({current_num_ids_from_data}) matches model's expected num_ids ({target_num_ids}). No padding needed.")
else:
    print("Could not determine target_num_ids reliably or no padding action taken. Proceeding with original data.")


/home/jkabir/Deep-Learning-Project/Data/one_csv

All CSVs now have 131 frames after trimming
Minimum records per ID: 2
Average records per ID: 100.29
Maximum records per ID: 131

Minimum IDs (Vehicles) per frame: 8
Average IDs (Vehicles) per frame: 13.02
Maximum IDs (Vehicles) per frame: 17

Before normalization:
X range: 85.0000 to 1907.0000
Y range: 609.0000 to 793.0000
Height range: 19.0000 to 178.0000
Width range: 21.0000 to 370.0000
Frame range: 0.0000 to 130.0000

After normalization:
X range: 0.0000 to 5.0000
Y range: 0.0000 to 5.0000
Height range: 0.0000 to 5.0000
Width range: 0.0000 to 5.0000
Frame range: 0.0000 to 5.0000
Determined tensor ID dimension size based on max(ID_Norm): 17
All data tensor shape: torch.Size([1, 131, 17, 5])
Data tensor shape: torch.Size([1, 131, 17, 5])
torch.Size([100, 17, 5])
torch.Size([100, 17, 5])
Initial Data X shape: torch.Size([2, 100, 17, 5]), Y shape: torch.Size([2, 30, 17, 2])
Number of IDs in current data: 17
Loaded model does not have 'nu

In [33]:
from Training.basicEval import test_model
from Training.customLoss import PaddedMSELoss

# test_model(model, test_loader, PaddedMSELoss(), xy_scaler)

In [35]:
import csv
from tqdm.notebook import tqdm
import numpy as np


def denorm_y(y, feature_scaler):
    y_flat = np.reshape(y, (y.shape[0] * y.shape[1], y.shape[2]))
    print(f"Y_flat shape: {y_flat.shape}")
    y_denorm_flat = feature_scaler.inverse_transform(y_flat)
    y_denorm = np.reshape(y_denorm_flat, (y.shape[0], y.shape[1], 2))
    print(f"Y_denorm shape: {y_denorm.shape}")
    print(f"Y_denorm: {y_denorm[0][0]}")
    return y_denorm
def denorm_x(x, frame_scaler, feature_scaler):
    x_flat = np.reshape(x, (x.shape[0] * x.shape[1], x.shape[2]))
    x_xy_only = x_flat[:, 1:3]
    x_xy_denorm_flat = feature_scaler.inverse_transform(x_xy_only)
    x_xy_denorm = np.reshape(x_xy_denorm_flat, (x.shape[0], x.shape[1], 2)) # shape (100, 18, 2)
    print(f"X_xy_denorm shape: {x_xy_denorm.shape}")
    print(f"X_flat shape: {x_flat.shape}")

    return x_xy_denorm

def predict_130_frames(model, X, Y, csv_path, frame_scaler, feature_scaler, sequence_idx, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    headers = ['Frame', 'ID', 'X_pred', 'Y_pred', 'X_true', 'Y_true']
    print(f"Exporting predictions to {csv_path}...")
    
    with open(csv_path, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(headers)
        
        with torch.no_grad():
            # X shape: (100, 18, 5)
            # Y shape: (30, 18, 2)
            
            # X contains (Seq_idx, ID, Features) 
            #    Features = [Frame, X, Y, Width, Height]
            # Y contains [Seq_idx, ID, X, Y]
            
            
            x = X[sequence_idx]
            x_unsqueezed = x.unsqueeze(0).to(device)
            y = Y[sequence_idx]
            y = y.cpu().numpy()
            y_pred = model(x_unsqueezed).squeeze(0).cpu().numpy()
            print(f"X shape: {x.shape}")
            print(f"Y shape: {y.shape}")
            print(f"Y_pred shape: {y_pred.shape}")
            
            print(f"First ID of First frame of X")
            print(x[0][0])
            print(f"First ID of First frame of Y")
            print(y[0][0])
            print(f"First ID of First frame of Y_pred")
            print(y_pred[0][0])
            
            x_denorm = denorm_x(x, frame_scaler, feature_scaler)
            y_denorm = denorm_y(y, feature_scaler)
            y_pred_denorm = denorm_y(y_pred, feature_scaler)
            
            print(f"First ID of First frame of X_denorm")
            print(x_denorm[0][0])
            print(f"First ID of First frame of Y_denorm")
            print(y_denorm[0][0])
            print(f"First ID of First frame of Y_pred_denorm")
            print(y_pred_denorm[0][0])
            
            # Write first 100 frames from x only
            # true = pred
            for frame_id, frame in enumerate(x_denorm, start=0):
                for v_id, v_id_features in enumerate(frame):
                    row = [
                        frame_id,  # Frame
                        v_id,  # ID
                        int(v_id_features[0]),  # X_pred
                        int(v_id_features[1]),  # Y_pred
                        int(v_id_features[0]),  # X_true
                        int(v_id_features[1])   # Y_true
                    ]
                    if np.any(v_id_features < 0):
                        continue
                    csv_writer.writerow(row)
            # Write next 30 frames from y and y_pred
            for seq_idx in range(30):
                y_frame = y_denorm[seq_idx]
                y_pred_frame = y_pred_denorm[seq_idx]
                for y_id, (y_id_features, y_pred_id_features) in enumerate(zip(y_frame, y_pred_frame)):
                    row = [
                        seq_idx + 100,  # Frame
                        y_id,
                        int(y_pred_id_features[0]),  # X_pred
                        int(y_pred_id_features[1]),  # Y_pred
                        int(y_id_features[0]),  # X_true
                        int(y_id_features[1]),   # Y_true
                    ]
                    if np.any(y_id_features < 0):
                        pass
                        continue
                    csv_writer.writerow(row)
    print(f"Finished exporting predictions to {csv_path}")
predict_130_frames(
    model,
    X,
    Y,
    'predictions.csv',
    frame_scaler,
    xy_scaler,
    sequence_idx=0,  # Change this to the desired sequence index
    device=None
)


Exporting predictions to predictions.csv...
X shape: torch.Size([100, 20, 5])
Y shape: (30, 20, 2)
Y_pred shape: (30, 20, 2)
First ID of First frame of X
tensor([0.0000, 4.8052, 2.2283, 2.1060, 3.0818])
First ID of First frame of Y
[-1. -1.]
First ID of First frame of Y_pred
[-0.84817743  0.23523465]
X_xy_denorm shape: (100, 20, 2)
X_flat shape: torch.Size([2000, 5])
Y_flat shape: (600, 2)
Y_denorm shape: (30, 20, 2)
Y_denorm: [-279.4  572.2]
Y_flat shape: (600, 2)
Y_denorm shape: (30, 20, 2)
Y_denorm: [-224.07585  617.6566 ]
First ID of First frame of X_denorm
[1835.99997311  690.9999958 ]
First ID of First frame of Y_denorm
[-279.4  572.2]
First ID of First frame of Y_pred_denorm
[-224.07585  617.6566 ]
Finished exporting predictions to predictions.csv
